# Destilar la voz **Alex** (Kokoro) a Piper
Timbre de Alex, sin grabar nada. Guarda en tu Google Drive y **retoma si Colab se corta**.
Entrenamiento: fork `KiON-GiON/piper1-gpl@fixes` (comandos verificados contra el código real).

**Antes:** Entorno de ejecución → Cambiar tipo → **T4 GPU**.
**Celda 1** prepara · **Celda 2** entrena (reejecutable) · **Celda 3** exporta y descarga.

In [ ]:
#@title 1. PREPARAR — reusa el dataset del Drive, muestra progreso y avisa con un tono
import IPython, torch, os, time, shutil
from IPython.display import Javascript, Audio, display

t0 = time.time()
def paso(msg):
    print(f"\n[{(time.time()-t0)/60:5.1f} min] === {msg} ===", flush=True)

# Anti-desconexion (clic automatico cada 60s)
display(Javascript('function _keep(){ var b=document.querySelector("colab-toolbar-button#connect"); if(b) b.click() } setInterval(_keep, 60000)'))

assert torch.cuda.is_available(), "Activa T4: Entorno de ejecucion -> Cambiar tipo de entorno -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
WORK = "/content/drive/MyDrive/LoudVox/alex"
os.makedirs(WORK, exist_ok=True)
print("Carpeta de trabajo en Drive:", WORK)

DATASET = "/content/dataset"
DATA_ZIP = WORK + "/dataset_alex.zip"

# --- 1) DATASET: si ya esta en el Drive, se recupera; si no, se genera y se guarda ---
if os.path.exists(DATA_ZIP):
    paso("Recuperando el dataset del Drive (NO regenero audios)")
    shutil.unpack_archive(DATA_ZIP, DATASET)
    print(f"Dataset recuperado del Drive: {len(os.listdir(DATASET + '/wavs'))} audios. (Te ahorras ~1 hora.)")
else:
    paso("Generando el dataset con la voz Alex (SOLO la primera vez, tarda ~1 h)")
    !pip install -q kokoro-onnx==0.5.0
    !wget -q -nc -O /content/kokoro.onnx "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx"
    !wget -q -nc -O /content/voices.bin "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin"
    !wget -q -O /content/corpus.txt "https://raw.githubusercontent.com/Lazy-Money/Loud-Web/claude/readvox-research-slumjx/colab/corpus/corpus_es_1300.txt"
    import wave, numpy as np
    from kokoro_onnx import Kokoro
    frases = [l.strip() for l in open("/content/corpus.txt", encoding="utf-8") if l.strip()]
    print(f"{len(frases)} frases a generar. Vas a ver el avance cada 50.")
    kokoro = Kokoro("/content/kokoro.onnx", "/content/voices.bin")
    os.makedirs(DATASET + "/wavs", exist_ok=True)
    rows, total, tg = [], 0.0, time.time()
    for i, f in enumerate(frases):
        if i and i % 50 == 0:
            el = time.time() - tg
            eta = el / i * (len(frases) - i) / 60
            print(f"  {i}/{len(frases)} frases  ·  {el/60:4.1f} min hechos  ·  faltan ~{eta:2.0f} min", flush=True)
        try:
            s, r = kokoro.create(f, voice="em_alex", speed=1.0, lang="es")
        except Exception:
            continue
        idx = np.linspace(0, len(s)-1, int(len(s)*22050/r))
        d = np.clip(np.interp(idx, np.arange(len(s)), s)*32767, -32768, 32767).astype(np.int16)
        if not 1.0 <= len(d)/22050 <= 20.0:
            continue
        n = f"f{i:05d}.wav"
        with wave.open(f"{DATASET}/wavs/{n}", "wb") as w:
            w.setnchannels(1); w.setsampwidth(2); w.setframerate(22050); w.writeframes(d.tobytes())
        rows.append(f"{n}|{f}"); total += len(d)/22050
    open(DATASET + "/metadata.csv", "w", encoding="utf-8").write("\n".join(rows) + "\n")
    print(f"Dataset generado: {len(rows)} clips, {total/60:.1f} min de audio.")
    paso("Guardando el dataset en tu Drive (para no regenerarlo NUNCA mas)")
    shutil.make_archive(WORK + "/dataset_alex", "zip", DATASET)
    print(f"Guardado en Drive: dataset_alex.zip ({os.path.getsize(DATA_ZIP)/1e6:.0f} MB)")

# --- 2) INSTALAR Piper (fork verificado), avisando cada paso ---
paso("Instalando paquetes del sistema (apt)")
!apt-get -q update -y > /dev/null 2>&1
!apt-get -q install -y build-essential cmake ninja-build espeak-ng aria2 > /dev/null 2>&1
paso("Clonando el repo de entrenamiento")
%cd /content
![ -d piper1-gpl ] || git clone -q -b fixes https://github.com/KiON-GiON/piper1-gpl.git
%cd /content/piper1-gpl
paso("Instalando piper con pip (~2-3 min, sin salida mientras trabaja)")
!python -m pip install -q -e .[train]
paso("Compilando el alineador y ajustando protobuf")
!bash build_monotonic_align.sh > /dev/null 2>&1
!pip install -q --upgrade gdown scikit-build protobuf==3.20.3
!python setup.py build_ext --inplace > /dev/null 2>&1

# --- 3) CHECKPOINT base: cacheado en Drive tambien ---
BASE_DRIVE = WORK + "/base.ckpt"
if os.path.exists(BASE_DRIVE) and os.path.getsize(BASE_DRIVE) > 10e6:
    paso("Recuperando el checkpoint base del Drive")
    shutil.copy(BASE_DRIVE, "/content/base.ckpt")
else:
    paso("Descargando el checkpoint base (espanol davefx) desde Hugging Face")
    from huggingface_hub import hf_hub_download
    _b = hf_hub_download(repo_id="rhasspy/piper-checkpoints", repo_type="dataset", filename="es/es_ES/davefx/medium/epoch=5629-step=1605020.ckpt")
    shutil.copy(_b, "/content/base.ckpt")
    shutil.copy("/content/base.ckpt", BASE_DRIVE)
_mb = os.path.getsize("/content/base.ckpt") / 1e6
print(f"base.ckpt: {_mb:.1f} MB")
assert _mb > 10, "La descarga del checkpoint base fallo. Volve a correr la celda 1."
torch.load("/content/base.ckpt", map_location="cpu", weights_only=False)
print("Checkpoint base OK.")

# --- LISTO + tono para avisarte ---
paso(f"LISTO en {(time.time()-t0)/60:.1f} min total. Corre la celda 2 para entrenar.")
import numpy as _np
_sr = 22050; _tt = _np.linspace(0, 0.7, int(_sr*0.7))
_beep = 0.3*_np.sin(2*_np.pi*880*_tt)*_np.exp(-2.5*_tt) + 0.2*_np.sin(2*_np.pi*1320*_tt)*_np.exp(-2.5*_tt)
display(Audio(_beep, rate=_sr, autoplay=True))

In [ ]:
#@title 2. ENTRENAR — backup rotativo de 2 slots en Drive (version_1=ultimo, version_0=respaldo). Reejecutable.
import os, glob, time, shutil, threading, torch

WORK   = "/content/drive/MyDrive/LoudVox/alex"
LOCAL  = "/content/train_local"          # checkpoints locales: rapido y confiable
BACKUP = WORK + "/backup"                 # SOLO 2 archivos aca, nunca mas
V1 = BACKUP + "/version_1.ckpt"           # el ULTIMO
V0 = BACKUP + "/version_0.ckpt"           # el ANTERIOR (respaldo por si version_1 falla)
os.makedirs(BACKUP, exist_ok=True)
MIN = 100_000_000                         # un checkpoint valido pesa cientos de MB

def _epoch(p):
    try: return int(torch.load(p, map_location="cpu", weights_only=False).get("epoch", -1))
    except Exception: return -1
def _ok(p): return os.path.exists(p) and os.path.getsize(p) > MIN

# --- (una sola vez) sembrar version_1 desde tu mejor checkpoint viejo ---------
if not _ok(V1):
    viejos = [p for p in glob.glob(WORK + "/lightning_logs/version_*/checkpoints/last.ckpt") if _ok(p)]
    if viejos:
        mejor = max(viejos, key=_epoch)
        print(f"Sembrando version_1 desde tu checkpoint mas entrenado (epoch {_epoch(mejor)})...")
        shutil.copy(mejor, V1)

# --- elegir desde donde retomar: version_1; si falla, promover version_0 ------
if _ok(V1):
    resume = V1; print(f"RETOMANDO desde version_1 (epoch {_epoch(V1)})")
    init = f'--ckpt_path "{V1}"'
elif _ok(V0):
    shutil.copy(V0, V1); resume = V1
    print(f"version_1 dañado -> promovido version_0 a version_1 (epoch {_epoch(V1)})")
    init = f'--ckpt_path "{V1}"'
else:
    resume = None; print("PRIMER entrenamiento: parto del checkpoint base español")
    init = '--model.init_from_checkpoint /content/base.ckpt'

# --- backup rotativo en 2do plano: cada guardado local rota a Drive -----------
_stop = threading.Event()
def _newest_local():
    c = glob.glob(LOCAL + "/lightning_logs/**/checkpoints/last.ckpt", recursive=True)
    return max(c, key=os.path.getmtime) if c else None
def _rotar(local):
    tmp = BACKUP + "/_incoming.ckpt"
    shutil.copy(local, tmp)                    # 1) subida COMPLETA a temp
    if os.path.exists(V1): os.replace(V1, V0)  # 2) version_1 -> version_0
    os.replace(tmp, V1)                        # 3) temp -> version_1
    print(f"  [backup {time.strftime('%H:%M:%S')}] Drive actualizado: version_1 nuevo, anterior -> version_0", flush=True)
def _watch():
    visto = 0
    while not _stop.is_set():
        try:
            loc = _newest_local()
            if loc and os.path.getmtime(loc) != visto:
                visto = os.path.getmtime(loc); _rotar(loc)
        except Exception as e:
            print("  [backup] reintenta:", e, flush=True)
        _stop.wait(20)
threading.Thread(target=_watch, daemon=True).start()

# --- entrenar (guarda LOCAL cada 5 epochs; el watcher lo sube a Drive) ---------
cmd = (
 'cd /content/piper1-gpl && python -m piper.train fit --data.voice_name "alex" '
 '--data.csv_path /content/dataset/metadata.csv --data.audio_dir /content/dataset/wavs '
 '--data.espeak_voice es --data.cache_dir /content/cache '
 f'--data.config_path "{WORK}/alex.onnx.json" --data.batch_size 12 --model.sample_rate 22050 '
 '--data.validation_split 0 --data.num_test_examples 0 '
 f'--trainer.default_root_dir "{LOCAL}" '
 '--trainer.accelerator gpu --trainer.devices 1 --trainer.max_epochs 10000 '
 '--trainer.precision 16-mixed --checkpoint.save_top_k 0 --checkpoint.monitor null '
 '--last_checkpoint.every_n_epochs 5 ' + init
)
try:
    get_ipython().system(cmd)
finally:
    _stop.set()
    loc = _newest_local()
    if loc:
        print("Backup final antes de cerrar..."); _rotar(loc)

In [ ]:
#@title 3. EXPORTAR Y DESCARGAR — exporta desde el ultimo backup (version_1)
import os, json
WORK = "/content/drive/MyDrive/LoudVox/alex"
V1 = WORK + "/backup/version_1.ckpt"; V0 = WORK + "/backup/version_0.ckpt"
ckpt = V1 if (os.path.exists(V1) and os.path.getsize(V1) > 100_000_000) else V0
assert os.path.exists(ckpt), "Todavia no hay backup. Deja correr la celda 2 unos epochs."
print("Exportando desde:", ckpt)
get_ipython().system(f'cd /content/piper1-gpl && python -m piper.train.export_onnx --checkpoint "{ckpt}" --output-file "{WORK}/alex.onnx"')
cfgp = WORK + "/alex.onnx.json"
if os.path.exists(cfgp):
    cfg = json.load(open(cfgp, encoding="utf-8")); cfg.setdefault("phoneme_map", {})
    json.dump(cfg, open(cfgp, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
from google.colab import files
files.download(WORK + "/alex.onnx"); files.download(cfgp)
print("Copia ambos a tu carpeta de voces de LoudVox y elegi 'alex' en Configuracion")